<a href="https://colab.research.google.com/github/priyal6/DL/blob/main/RNN_Pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
import pandas as pd

df = pd.read_csv("/content/100_Unique_QA_Dataset.csv")

df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [36]:
def tokenize(text):
  text = text.lower()
  text = text.replace('?', '')
  text = text.replace("'", "")
  return text.split()

In [37]:
tokenize('What is the capital of France?')

['what', 'is', 'the', 'capital', 'of', 'france']

In [38]:
vocab = {'<UNK>':0}

In [39]:
def build_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])

  merged_tokens = tokenized_question + tokenized_answer

  for token in merged_tokens:

    if token not in vocab:
      vocab[token] = len(vocab)

In [40]:
df.apply(build_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [41]:
def text_to_indices(text, vocab):
  indexed_text = []

  for token in tokenize(text):

    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])
  return indexed_text

In [42]:
text_to_indices("What is campusx", vocab)

[1, 2, 0]

In [43]:
import torch
from torch.utils.data import Dataset, DataLoader

In [44]:
class QADataset(Dataset):

  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):

    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [45]:
dataset = QADataset(df, vocab)

In [46]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [47]:
for question, answer in dataloader:
  print(question, answer[0])

tensor([[  1,   2,   3, 146,  86,  19, 192, 193]]) tensor([194])
tensor([[10, 75, 76]]) tensor([77])
tensor([[ 10,   2,  62,  63,   3, 283,   5, 284]]) tensor([285])
tensor([[ 78,  79, 150, 151,  14, 152, 153]]) tensor([154])
tensor([[ 1,  2,  3, 50, 51, 19,  3, 45]]) tensor([52])
tensor([[ 42, 117, 118,   3, 119,  94, 120]]) tensor([121])
tensor([[ 42, 137,   2,  62,  39,   3, 322, 323]]) tensor([6])
tensor([[ 42, 299, 300, 118,  14, 301, 302, 158, 303, 304, 305, 306]]) tensor([307])
tensor([[ 42, 137,   2, 138,  39, 175, 269]]) tensor([99])
tensor([[  1,   2,   3,  69,   5, 155]]) tensor([156])
tensor([[10, 55,  3, 56,  5, 57]]) tensor([58])
tensor([[  1,   2,   3,   4,   5, 286]]) tensor([287])
tensor([[  1,   2,   3,  33,  34,   5, 245]]) tensor([246])
tensor([[ 42, 250, 251, 118, 252, 253]]) tensor([254])
tensor([[ 10, 140,   3, 141, 171,   5,   3,  70, 172]]) tensor([173])
tensor([[  1,   2,   3,   4,   5, 109]]) tensor([317])
tensor([[ 1,  2,  3,  4,  5, 73]]) tensor([74])
tenso

In [48]:
import torch.nn as nn

In [49]:
class SimpleRNN(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim = 50)
    self.rnn = nn.RNN(50,64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))

    return output

In [50]:
learning_rate = 0.001
epochs = 20

In [51]:
model = SimpleRNN(len(vocab))

In [52]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [53]:
#training loop

for epoch in range(epochs):

  total_loss = 0

  for question, answer in dataloader:
    optimizer.zero_grad()

    output = model(question)

    loss = criterion(output, answer[0])

    loss.backward()

    optimizer.step()

    total_loss = total_loss + loss.item()

print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 20, Loss: 10.986336


In [56]:
def predict(model, question, threshold = 0.5):
  numerical_question = text_to_indices(question, vocab)

  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  output = model(question_tensor)

  probs = torch.nn.functional.softmax(output, dim=1)

  value, index = torch.max(probs, dim=1)

  if value< threshold:
    print("I dont know")
  print(list(vocab.keys())[index])

In [57]:
predict(model, "What is the largest planet in our solar system?")

jupiter


In [58]:
list(vocab.keys())[7]

'paris'